In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score
import sys
sys.path.append('..')  # если utils лежит на уровень выше
from utils.extract_features import extract_features

# Загружаем общий файл с данными (у тебя он называется, допустим, XSS_dataset.csv)
df = pd.read_csv("../data/processed/balanced_unified_dataset.csv")  # поменяй путь, если нужно
print("Всего строк:", len(df))
print("Источники:", df['source'].unique())

Всего строк: 70032
Источники: ['fwaf' 'csic2010' 'vulnxss' 'kaggle' 'httpparams' 'payloadbox']


In [11]:
def tpr_at_1pct_fpr(y_true, y_pred_proba):
    """
    Считает долю правильно пойманных атак (TPR) при 1% ложных срабатываний (FPR).
    y_true: массив 0/1 (1 — атака)
    y_pred_proba: вероятности класса 1
    """
    # Сортируем по убыванию вероятности
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    sorted_true = y_true[sorted_indices]
    
    total_neg = np.sum(y_true == 0)
    if total_neg == 0:
        return 0.0  # если нет нормальных запросов, то не считаем
    
    target_fp = int(0.01 * total_neg)  # сколько ложных срабатываний допустимо (1%)
    fp = 0
    tp = 0
    for label in sorted_true:
        if label == 0:
            fp += 1
        else:
            tp += 1
        if fp >= target_fp:
            break
    
    total_pos = np.sum(y_true == 1)
    if total_pos == 0:
        return 0.0
    return tp / total_pos

In [12]:
print("Извлечение признаков для всех строк...")
features_list = []
for i, text in enumerate(df['text']):
    if i % 1000 == 0:
        print(f"Обработано {i}/{len(df)}")
    features_list.append(extract_features(text, False))  # False — как в твоём коде

X_all = pd.DataFrame(features_list)
y_all = df['label'].values
sources = df['source'].values

print("Признаки извлечены. Размер:", X_all.shape)

Извлечение признаков для всех строк...
Обработано 0/70032
Обработано 1000/70032
Обработано 2000/70032
Обработано 3000/70032
Обработано 4000/70032
Обработано 5000/70032
Обработано 6000/70032
Обработано 7000/70032
Обработано 8000/70032
Обработано 9000/70032
Обработано 10000/70032
Обработано 11000/70032
Обработано 12000/70032
Обработано 13000/70032
Обработано 14000/70032
Обработано 15000/70032
Обработано 16000/70032
Обработано 17000/70032
Обработано 18000/70032
Обработано 19000/70032
Обработано 20000/70032
Обработано 21000/70032
Обработано 22000/70032
Обработано 23000/70032
Обработано 24000/70032
Обработано 25000/70032
Обработано 26000/70032
Обработано 27000/70032
Обработано 28000/70032
Обработано 29000/70032
Обработано 30000/70032
Обработано 31000/70032
Обработано 32000/70032
Обработано 33000/70032
Обработано 34000/70032
Обработано 35000/70032
Обработано 36000/70032
Обработано 37000/70032
Обработано 38000/70032
Обработано 39000/70032
Обработано 40000/70032
Обработано 41000/70032
Обработа

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
results = {}

# Список всех источников (порядок не важен)
source_names = ['fwaf', 'csic2010', 'kaggle', 'vulnxss', 'httpparams']

for test_source in source_names:
    print(f"\n===== Тестируем на {test_source} =====")
    
    # Индексы строк, которые принадлежат этому источнику (тест)
    test_mask = (sources == test_source)
    X_test = X_all[test_mask].values
    y_test = y_all[test_mask]
    
    # Обучающие данные — все остальные
    train_mask = ~test_mask
    X_train = X_all[train_mask].values
    y_train = y_all[train_mask]
    
    # Масштабируем
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Обучаем логистическую регрессию (с балансировкой классов)
    model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train)
    
    # Предсказания
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    
    # Считаем метрики
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred_binary)
    acc = accuracy_score(y_test, y_pred_binary)
    tpr = tpr_at_1pct_fpr(y_test, y_pred_proba)
    
    # Сохраняем
    results[test_source] = {
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'f1': f1,
        'accuracy': acc,
        'tpr@1%fpr': tpr
    }
    
    print(f"ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}, F1: {f1:.4f}, Accuracy: {acc:.4f}, TPR@1%FPR: {tpr:.4f}")


===== Тестируем на fwaf =====


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


ROC-AUC: nan, PR-AUC: 0.0000, F1: 0.0000, Accuracy: 0.8280, TPR@1%FPR: 0.0000

===== Тестируем на csic2010 =====


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.0270, Accuracy: 0.0137, TPR@1%FPR: 0.0000

===== Тестируем на kaggle =====
ROC-AUC: 0.4786, PR-AUC: 0.9765, F1: 0.9878, Accuracy: 0.9760, TPR@1%FPR: 0.0001

===== Тестируем на vulnxss =====


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.9602, Accuracy: 0.9234, TPR@1%FPR: 0.0000

===== Тестируем на httpparams =====
ROC-AUC: 0.9237, PR-AUC: 0.6564, F1: 0.6988, Accuracy: 0.8111, TPR@1%FPR: 0.0343


In [17]:
result_df = pd.DataFrame(results).T  # транспонируем, чтобы источники были строками
print("\n=== Итоговая таблица LODO ===")
print(result_df.round(4))

# Сохраним в CSV, если нужно
result_df.to_csv("lodo_results_lr.csv")
print("Результаты сохранены в lodo_results_lr.csv")


=== Итоговая таблица LODO ===
            roc_auc  pr_auc      f1  accuracy  tpr@1%fpr
fwaf            NaN  0.0000  0.0000    0.8280     0.0000
csic2010        NaN  1.0000  0.0270    0.0137     0.0000
kaggle       0.4786  0.9765  0.9878    0.9760     0.0001
vulnxss         NaN  1.0000  0.9602    0.9234     0.0000
httpparams   0.9237  0.6564  0.6988    0.8111     0.0343
Результаты сохранены в lodo_results_lr.csv
